# Session 2: Data Preparation Pipeline for a Simple Language Model

This notebook covers **Session 2 only**.

Goal: convert raw text into training data for a simple next-word prediction model.

Pipeline:

```text
raw text -> tokens -> vocabulary -> context-next word pairs -> numerical features -> labels
```

We will use a short public-domain excerpt from **Alice's Adventures in Wonderland** as a well-known text source.

By the end of this notebook, you should understand how to build:

- `X`: input feature matrix
- `y`: target label vector

These will be used later to train a Feedforward Neural Network.

## 1. From Raw Text to Training Data

In Session 1, we learned the concept:

```text
context words -> next word
```

In Session 2, we prepare real training examples.

A machine learning model cannot directly learn from raw text like this:

```text
Alice was beginning to get very tired
```

We need to transform the text step by step:

```text
raw text
-> tokens
-> vocabulary
-> context-next word pairs
-> numerical input features
-> target labels
```

## 2. Load or Define a Text Corpus

A **corpus** is a collection of text used for training or analysis.

For this lesson, we use a short excerpt inspired by the beginning of **Alice's Adventures in Wonderland**.

We keep the corpus small so every step can be inspected clearly.

In [ ]:
corpus = """
Alice was beginning to get very tired of sitting by her sister on the bank.
Alice had nothing to do.
Once or twice she had peeped into the book her sister was reading.
The book had no pictures or conversations.
Alice thought a book without pictures or conversations was not useful.
Alice was very tired and sleepy.
"""

print(corpus)

## 3. Tokenize the Corpus

Tokenization means splitting text into tokens.

For this lesson:

```text
one word = one token
```

We also clean the text by:

- converting to lowercase,
- removing punctuation,
- keeping only word tokens.

This gives us more consistent tokens.

In [ ]:
import re


def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return tokens


tokens = tokenize(corpus)

print("Number of tokens:", len(tokens))
print(tokens)

### Why clean punctuation?

Without cleaning:

```text
bank.
```

and:

```text
bank
```

would look like different tokens.

For a beginner language model, we usually want them to be the same word.

In [ ]:
simple_split_tokens = corpus.lower().split()

print("Using simple split:")
print(simple_split_tokens[:15])

print("\nUsing regex tokenizer:")
print(tokens[:15])

## 4. Build Vocabulary from the Corpus

The vocabulary is the list of unique words in the corpus.

We use:

```python
sorted(set(tokens))
```

so the vocabulary order is stable and everyone gets the same word IDs.

In [ ]:
vocabulary = sorted(set(tokens))

print("Vocabulary size:", len(vocabulary))
print(vocabulary)

Now create two dictionaries:

- `word_to_id`: convert word to number
- `id_to_word`: convert number back to word

These mappings are needed when converting between text and numerical labels.

In [ ]:
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = {index: word for word, index in word_to_id.items()}

print("word_to_id:")
print(word_to_id)

print("\nid_to_word:")
print(id_to_word)

## 5. Create Context-Target Pairs with Sliding Window

We now create training examples.

Use:

```python
context_size = 2
```

That means:

```text
two previous words -> next word
```

Example:

```text
alice was -> beginning
was beginning -> to
beginning to -> get
```

In [ ]:
context_size = 2

pairs = []

for i in range(len(tokens) - context_size):
    context = tokens[i:i + context_size]
    target = tokens[i + context_size]
    pairs.append((context, target))

print("Number of context-target pairs:", len(pairs))
print()

for context, target in pairs[:20]:
    print(context, "->", target)

### Why is it called a sliding window?

The context window moves one token at a time.

For tokens:

```python
['alice', 'was', 'beginning', 'to', 'get']
```

The windows are:

```text
alice was -> beginning
was beginning -> to
beginning to -> get
```

In [ ]:
demo_tokens = tokens[:8]

print("Demo tokens:", demo_tokens)
print()

for i in range(len(demo_tokens) - context_size):
    context = demo_tokens[i:i + context_size]
    target = demo_tokens[i + context_size]
    print("start index", i, ":", context, "->", target)

## 6. Convert Context Words to Numerical Features

Machine learning models need numbers.

We have context words like:

```python
['alice', 'was']
```

But the model needs numerical features.

A tempting approach is:

```python
['alice', 'was'] -> [word_id_of_alice, word_id_of_was]
```

This is easy, but it has a problem: the model may treat word IDs as ordered numerical values.

For example, ID 20 is not really greater than ID 3 in meaning.

So for this lesson, we use **one-hot encoding**.

In [ ]:
example_context = pairs[0][0]
example_context_ids = [word_to_id[word] for word in example_context]

print("Example context:", example_context)
print("Context as word IDs:", example_context_ids)

## 7. One-Hot Encoding for Context Words

One-hot encoding represents one word as a vector.

If the vocabulary is:

```python
['alice', 'and', 'bank', 'book']
```

then:

```text
alice -> [1, 0, 0, 0]
book  -> [0, 0, 0, 1]
```

Only one position is `1`. All other positions are `0`.

In [ ]:
import numpy as np


def one_hot(word, word_to_id):
    vector = np.zeros(len(word_to_id), dtype=int)
    vector[word_to_id[word]] = 1
    return vector


word = "alice"
vector = one_hot(word, word_to_id)

print("Word:", word)
print("Word ID:", word_to_id[word])
print("One-hot vector:")
print(vector)
print("Vector length:", len(vector))

For a context of two words, we create one one-hot vector for each word, then **concatenate** them.

Example:

```text
['alice', 'was']
```

becomes:

```text
concatenate(one_hot('alice'), one_hot('was'))
```

If vocabulary size is 36 and context size is 2, the feature vector length is:

```text
36 * 2 = 72
```

In [ ]:
def context_to_vector(context, word_to_id, context_size):
    vocab_size = len(word_to_id)
    vector = np.zeros(context_size * vocab_size, dtype=int)

    for position, word in enumerate(context):
        word_id = word_to_id[word]
        feature_index = position * vocab_size + word_id
        vector[feature_index] = 1

    return vector


example_context = ["alice", "was"]
example_vector = context_to_vector(example_context, word_to_id, context_size)

print("Context:", example_context)
print("Vocabulary size:", len(word_to_id))
print("Context size:", context_size)
print("Expected feature vector length:", len(word_to_id) * context_size)
print("Feature vector length:", len(example_vector))
print("Active positions:", np.where(example_vector == 1)[0].tolist())
print(example_vector)

## 8. Build Input Matrix `X`

`X` contains all input feature vectors.

Each row is one context.

If we have 55 context-target pairs, then `X` has 55 rows.

The number of columns depends on:

```text
context_size * vocabulary_size
```

In [ ]:
X = np.array([
    context_to_vector(context, word_to_id, context_size)
    for context, target in pairs
])

print("X shape:", X.shape)
print("Number of rows:", X.shape[0])
print("Number of columns:", X.shape[1])

## 9. Build Target Vector `y`

`y` contains the target labels.

Each target word becomes a class ID.

Example:

```text
beginning -> 4
to -> 30
get -> 9
```

The exact IDs depend on the vocabulary order.

In [ ]:
y = np.array([
    word_to_id[target]
    for context, target in pairs
])

print("y shape:", y.shape)
print("First 10 target IDs:", y[:10])
print("First 10 target words:", [id_to_word[target_id] for target_id in y[:10]])

## 10. Inspect Dataset Shapes

For supervised learning, the number of rows in `X` and `y` must match.

Each row in `X` has one answer in `y`.

```text
X[i] -> y[i]
```

In [ ]:
print("Number of X rows:", len(X))
print("Number of y labels:", len(y))
print("Do they match?", len(X) == len(y))

print("\nContext size:", context_size)
print("Vocabulary size:", len(vocabulary))
print("Expected number of X columns:", context_size * len(vocabulary))
print("Actual number of X columns:", X.shape[1])

## 11. Preview Final Training Examples

Now inspect the final training data.

For each example:

- show context words,
- show target word,
- show target ID,
- show where the `1` values appear in the feature vector.

The positions of `1` tell us which words are active in the context.

In [ ]:
for index in range(8):
    context, target = pairs[index]
    feature_vector = X[index]
    active_positions = np.where(feature_vector == 1)[0].tolist()

    print("Example", index)
    print("Context:", context)
    print("Target word:", target)
    print("Target ID:", y[index])
    print("Active feature positions:", active_positions)
    print("---")

### Decode Active Feature Positions

The feature vector is divided into sections.

With context size 2:

```text
section 1 = first context word
section 2 = second context word
```

We can decode the active positions back into words.

In [ ]:
def decode_context_vector(vector, id_to_word, context_size):
    vocab_size = len(id_to_word)
    decoded_words = []

    for position in range(context_size):
        start = position * vocab_size
        end = start + vocab_size
        section = vector[start:end]
        word_id = int(np.argmax(section))
        decoded_words.append(id_to_word[word_id])

    return decoded_words


for index in range(5):
    decoded_context = decode_context_vector(X[index], id_to_word, context_size)
    print("Original context:", pairs[index][0])
    print("Decoded context:", decoded_context)
    print()

## 12. Common Data Preparation Mistakes

### Mistake 1: Unknown words

If a word is not in the vocabulary, `word_to_id[word]` will fail.

### Mistake 2: Inconsistent vocabulary order

If two notebooks use different vocabulary orders, their word IDs may be different.

Use `sorted(set(tokens))` to keep the order stable.

### Mistake 3: Wrong context size

The model expects a fixed input size.

If the model was prepared with context size 2, every input must use 2 context words.

### Mistake 4: Mismatched `X` and `y`

Every input row in `X` must have exactly one target label in `y`.

### Mistake 5: Treating raw word IDs as meaningful numbers

Word ID `20` is not more meaningful than word ID `3`.

That is why one-hot encoding is clearer for this beginner FNN lesson.

In [ ]:
# Mistake 1 demonstration: unknown word
unknown_word = "rabbit"

if unknown_word in word_to_id:
    print(word_to_id[unknown_word])
else:
    print("Unknown word:", unknown_word)
    print("This word is not in the vocabulary.")

In [ ]:
# Mistake 3 demonstration: wrong context size
wrong_context = ["alice", "was", "beginning"]

print("Expected context size:", context_size)
print("Given context size:", len(wrong_context))

if len(wrong_context) != context_size:
    print("Problem: this context has the wrong size for our prepared dataset.")

## Session 2 Summary

In this session, we prepared data for a simple language model.

We converted:

```text
raw text
```

into:

```text
X = input feature matrix
y = target label vector
```

Full pipeline:

```text
corpus
-> tokens
-> vocabulary
-> word IDs
-> context-target pairs
-> one-hot context vectors
-> X and y
```

In the next session, we can train a Feedforward Neural Network using:

```python
model.fit(X, y)
```